In [1]:
# Load the autoreload extension
%load_ext autoreload
# Set autoreload to reload all modules before executing code
%autoreload 2

In [2]:
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path 
from tqdm import tqdm
from copy import deepcopy
import pandas as pd 

from utils import *
set_style()

mode = 'sentinel' # 'venus' or 'sentinel'

base_venus = '/Data_large/marine/Datasets/VENuS/ds_L0/perfect'
base_sentinel = '/Data_large/marine/Datasets/VDS2Raw/imgs/'

base_path = base_venus if mode == 'venus' else base_sentinel

print('Listing the number of imgs:')
imgs = list(Path(base_path).glob('*.tif'))
print(len(imgs))

train_coco_path = '/Data_large/marine/Datasets/VENuS/annotations/perfect/train.json' if mode == 'venus' else '/Data_large/marine/Datasets/VDS2Raw/annotations/train.json'
val_coco_path = '/Data_large/marine/Datasets/VENuS/annotations/perfect/val.json' if mode == 'venus' else '/Data_large/marine/Datasets/VDS2Raw/annotations/val.json'
test_coco_path = '/Data_large/marine/Datasets/VENuS/annotations/perfect/test.json' if mode == 'venus' else '/Data_large/marine/Datasets/VDS2Raw/annotations/test.json'

train_coco = get_coco(train_coco_path)
val_coco = get_coco(val_coco_path)
test_coco = get_coco(test_coco_path)

annotations_train = train_coco['annotations']
annotations_val = val_coco['annotations']
annotations_test = test_coco['annotations']
annotations = annotations_train + annotations_val + annotations_test

def annotParserSen(x):
    try:
        return {'image_id':x['image_id'],'bbox':x['bbox'],'vessel_type':x['info'][0]['Ship type']}
    except KeyError:
        return {'image_id':x['image_id'],'bbox':x['bbox'],'vessel_type':[{'Ship type':'Unknown'}]}


images_train = train_coco['images']
images_val = val_coco['images']
images_test = test_coco['images']
images = images_train + images_val + images_test

imgidToImage = {x['id']:x['file_name'] for x in images} # dict because this is unique assignment
NameToID = {x['id']:x['file_name'] for x in images}

imgidAndBBox = [(x['image_id'],x['bbox']) for x in annotations] # list because assignment can be multiple
if mode == 'venus':
    imgidAndBBoxAndType = [(x['image_id'],x['bbox'],x['vessel_type']) for x in annotations] # list because assignment can be multiple
else:
    imgidAndBBoxAndType = [(annotParserSen(x)['image_id'],annotParserSen(x)['bbox'],annotParserSen(x)['vessel_type']) for x in annotations] # list because assignment can be multiple
    
imgidToBBox = convert_to_dict(imgidAndBBox)
filename_bbox = {imgidToImage[x]:y for x,y in imgidToBBox.items()}

nameToPaths = {x.name:x.as_posix() for x in imgs}
PathsToName = {x.as_posix():x.name for x in imgs}

Listing the number of imgs:
389


### Adding a buffer w 20 px

In [17]:
def add_buffer(x, y, width, height, buffer=5):
    x1 = max(0, x - buffer//2)
    y1 = max(0, y - buffer//2)
    x2 = width + buffer
    y2 = height + buffer
    return x1, y1, x2, y2

In [37]:
VERBOSE = False
# Initialize the dictionary:
band_indices = list(range(1,13,1)) if mode == 'venus' else [2,3,4,8]
band_index_to_tiff_index = {2:1, 3:2, 4:3, 8:4} # sentinel bands
tiff_index_to_band_index = {1:2, 2:3, 3:4, 4:8} # sentinel bands
# annotations_x_band = {band_index_to_tiff_index[i]:[] for i in band_indices} # initialize the dictionary
annotations_x_band = {i:[] for i in band_indices} # initialize the dictionary


### INIT:

dataset_type = {'train': train_coco, 'val': val_coco, 'test': test_coco}
picked_dataset = 'test'
for picked_dataset in ['train', 'val', 'test']:
    DATASET = deepcopy(dataset_type[picked_dataset])

    for idx, item in enumerate(tqdm(DATASET['annotations'], desc="Processing annotations")):
        # deepcopy the item
        
        annotation_id = item['id']
        image_id = item['image_id']
        bbox = item['bbox']
        area = item['area']
        
        ## 1. Read the image
        ### Get image name and path
        name = imgidToImage[image_id]
        stem = name.split('.')[0]
        path = nameToPaths[name]
        
        if VERBOSE:
            print(f'Processing {name}')
            print(f'bbox: {bbox}')
            print(f'area: {area}')
            print(f'path: {path}')
        
        img = read_tif(file_path=path, band_indices=list(range(1, len(band_indices)+1,1))) # read all the bands from the image
        
        # 2. Crop Img using bbox info  3. Save crops in the folders
        loop_indices = band_indices if mode == 'venus' else [band_index_to_tiff_index[i] for i in band_indices]
        for sel_band in loop_indices:
            x, y, width, height = bbox
            x, y, width, height = int(x), int(y), int(width), int(height)
            # add a buffer to the bbox
            x, y, width, height = add_buffer(x, y, width, height)
            bbox = (x, y, width, height)
            # Computing offssets:
            bandImg = img[0][sel_band]
            if VERBOSE:
                print(bandImg)
                print(f'bandImg: {bandImg.shape}')
            cropped_array = bandImg[y:y+height, x:x+width]
            img_x_method = {}
            for method in ('otsu', 'li', 'isodata', 'mean'):
                try:
                    img_x_method[method] = threshold(cropped_array, method=method)
                except:
                    img_x_method[method] = np.zeros_like(cropped_array, dtype=bool)
                    print(f'Error in {method}')

            # Combine masks where at least two agree
            combined_mask = np.zeros_like(cropped_array, dtype=bool)
            mask_count = sum(img_x_method.values())
            combined_mask[mask_count >= 3] = True
            # if no foreground pixels, use the old bbox
            if combined_mask.sum() == 0:
                new_bbox = deepcopy(bbox)
                print(f'No foreground pixels in {name}')
            else:
                dist = calculate_fit_distances(combined_mask) # calculate the distances
                print(f'dist: {dist}')
                # Updating bbox with distances
                new_bbox = update_bbox_with_dist(bbox, dist)
            
            if VERBOSE:
                print(f'bbox: {bbox}')
                print(f'new_bbox: {new_bbox}')
            
            updated_item = deepcopy(item)
            updated_item['bbox'] = new_bbox
            updated_item['area'] = new_bbox[2]*new_bbox[3]
            
            if updated_item['area'] <= 50:
                annotations_x_band[tiff_index_to_band_index[sel_band]].append(item)      
            else:
                # Inserting annotation in the dictionary per band
                annotations_x_band[tiff_index_to_band_index[sel_band]].append(updated_item)     
                
                
        # if idx == 5:
        #     break
                
                
    # Save the annotations
    senSave = '/Data_large/marine/Datasets/VDS2Raw/annotations/'
    venSave = '/Data_large/marine/Datasets/VENuS/annotations/perfect/'
    pickSave = venSave if mode == 'venus' else senSave
    save_path = f'{pickSave}{picked_dataset}_x_band.pkl'
    pd.to_pickle(annotations_x_band, save_path)

Processing annotations:   0%|          | 0/483 [00:00<?, ?it/s]

Processing annotations:   1%|          | 4/483 [00:00<00:14, 33.80it/s]

dist: {'top': 8, 'left': 4, 'right': 5, 'bottom': 4}
dist: {'top': 1, 'left': 7, 'right': 6, 'bottom': 10}
dist: {'top': 0, 'left': 9, 'right': 9, 'bottom': 15}
dist: {'top': 5, 'left': 11, 'right': 12, 'bottom': 14}
dist: {'top': 6, 'left': 2, 'right': 4, 'bottom': 0}
dist: {'top': 6, 'left': 5, 'right': 6, 'bottom': 6}
dist: {'top': 6, 'left': 9, 'right': 9, 'bottom': 16}
dist: {'top': 11, 'left': 11, 'right': 12, 'bottom': 18}
dist: {'top': 7, 'left': 4, 'right': 0, 'bottom': 3}
dist: {'top': 2, 'left': 6, 'right': 0, 'bottom': 8}
dist: {'top': 0, 'left': 0, 'right': 0, 'bottom': 5}
dist: {'top': 1, 'left': 0, 'right': 2, 'bottom': 1}
dist: {'top': 5, 'left': 1, 'right': 5, 'bottom': 0}
dist: {'top': 2, 'left': 0, 'right': 1, 'bottom': 0}
dist: {'top': 0, 'left': 0, 'right': 3, 'bottom': 5}
dist: {'top': 0, 'left': 1, 'right': 1, 'bottom': 0}
dist: {'top': 4, 'left': 2, 'right': 0, 'bottom': 3}
dist: {'top': 1, 'left': 4, 'right': 0, 'bottom': 9}
dist: {'top': 0, 'left': 6, 'right':

Processing annotations:   2%|▏         | 12/483 [00:00<00:13, 33.68it/s]

dist: {'top': 8, 'left': 3, 'right': 7, 'bottom': 0}
dist: {'top': 7, 'left': 3, 'right': 7, 'bottom': 0}
dist: {'top': 7, 'left': 3, 'right': 11, 'bottom': 0}
dist: {'top': 12, 'left': 4, 'right': 14, 'bottom': 0}
dist: {'top': 0, 'left': 2, 'right': 4, 'bottom': 3}
dist: {'top': 0, 'left': 5, 'right': 6, 'bottom': 10}
dist: {'top': 0, 'left': 6, 'right': 10, 'bottom': 13}
dist: {'top': 0, 'left': 10, 'right': 10, 'bottom': 11}
dist: {'top': 3, 'left': 6, 'right': 1, 'bottom': 4}
dist: {'top': 0, 'left': 3, 'right': 6, 'bottom': 0}
dist: {'top': 0, 'left': 9, 'right': 7, 'bottom': 15}
dist: {'top': 2, 'left': 11, 'right': 10, 'bottom': 14}
dist: {'top': 0, 'left': 2, 'right': 1, 'bottom': 0}
dist: {'top': 0, 'left': 0, 'right': 1, 'bottom': 0}
dist: {'top': 1, 'left': 1, 'right': 0, 'bottom': 0}
dist: {'top': 0, 'left': 0, 'right': 0, 'bottom': 0}
dist: {'top': 1, 'left': 4, 'right': 2, 'bottom': 6}
dist: {'top': 6, 'left': 5, 'right': 6, 'bottom': 6}
dist: {'top': 9, 'left': 6, 'righ

...

dist: {'top': 7, 'left': 0, 'right': 5, 'bottom': 1}
dist: {'top': 11, 'left': 0, 'right': 8, 'bottom': 1}
dist: {'top': 14, 'left': 1, 'right': 11, 'bottom': 2}
dist: {'top': 13, 'left': 0, 'right': 13, 'bottom': 5}
dist: {'top': 0, 'left': 0, 'right': 4, 'bottom': 0}
dist: {'top': 1, 'left': 0, 'right': 4, 'bottom': 0}
dist: {'top': 2, 'left': 0, 'right': 2, 'bottom': 0}
dist: {'top': 1, 'left': 0, 'right': 1, 'bottom': 0}
dist: {'top': 9, 'left': 3, 'right': 3, 'bottom': 0}
dist: {'top': 7, 'left': 5, 'right': 0, 'bottom': 0}
dist: {'top': 7, 'left': 7, 'right': 0, 'bottom': 0}
dist: {'top': 13, 'left': 4, 'right': 0, 'bottom': 0}
dist: {'top': 8, 'left': 4, 'right': 0, 'bottom': 3}
dist: {'top': 7, 'left': 5, 'right': 0, 'bottom': 6}
dist: {'top': 7, 'left': 8, 'right': 0, 'bottom': 12}
dist: {'top': 13, 'left': 0, 'right': 0, 'bottom': 0}
dist: {'top': 0, 'left': 4, 'right': 0, 'bottom': 0}
dist: {'top': 0, 'left': 5, 'right': 0, 'bottom': 3}
dist: {'top': 0, 'left': 4, 'right': 0

Processing annotations:  90%|█████████ | 84/93 [00:02<00:00, 35.21it/s]

dist: {'top': 0, 'left': 7, 'right': 0, 'bottom': 10}
dist: {'top': 0, 'left': 12, 'right': 0, 'bottom': 16}
dist: {'top': 0, 'left': 2, 'right': 6, 'bottom': 2}
dist: {'top': 0, 'left': 5, 'right': 7, 'bottom': 8}
dist: {'top': 0, 'left': 5, 'right': 10, 'bottom': 13}
dist: {'top': 0, 'left': 5, 'right': 14, 'bottom': 11}
dist: {'top': 0, 'left': 0, 'right': 5, 'bottom': 4}
dist: {'top': 0, 'left': 1, 'right': 7, 'bottom': 10}
dist: {'top': 0, 'left': 2, 'right': 10, 'bottom': 15}
dist: {'top': 0, 'left': 2, 'right': 12, 'bottom': 14}
dist: {'top': 0, 'left': 0, 'right': 1, 'bottom': 0}
dist: {'top': 0, 'left': 0, 'right': 0, 'bottom': 0}
dist: {'top': 0, 'left': 0, 'right': 0, 'bottom': 0}
dist: {'top': 0, 'left': 0, 'right': 0, 'bottom': 0}
dist: {'top': 7, 'left': 2, 'right': 3, 'bottom': 0}
dist: {'top': 6, 'left': 3, 'right': 2, 'bottom': 0}
dist: {'top': 6, 'left': 7, 'right': 7, 'bottom': 6}
dist: {'top': 11, 'left': 7, 'right': 10, 'bottom': 4}
dist: {'top': 0, 'left': 2, 'rig

Processing annotations: 100%|██████████| 93/93 [00:02<00:00, 34.10it/s]

dist: {'top': 0, 'left': 0, 'right': 5, 'bottom': 6}
dist: {'top': 0, 'left': 0, 'right': 7, 'bottom': 6}
dist: {'top': 4, 'left': 0, 'right': 12, 'bottom': 0}
dist: {'top': 0, 'left': 2, 'right': 13, 'bottom': 14}
dist: {'top': 4, 'left': 5, 'right': 2, 'bottom': 0}
dist: {'top': 8, 'left': 7, 'right': 3, 'bottom': 0}
dist: {'top': 20, 'left': 7, 'right': 11, 'bottom': 0}
dist: {'top': 11, 'left': 10, 'right': 3, 'bottom': 0}
dist: {'top': 3, 'left': 3, 'right': 5, 'bottom': 1}
dist: {'top': 6, 'left': 6, 'right': 7, 'bottom': 2}
dist: {'top': 7, 'left': 7, 'right': 10, 'bottom': 5}
dist: {'top': 9, 'left': 9, 'right': 12, 'bottom': 9}
dist: {'top': 2, 'left': 6, 'right': 0, 'bottom': 4}
dist: {'top': 3, 'left': 7, 'right': 0, 'bottom': 6}
dist: {'top': 3, 'left': 9, 'right': 0, 'bottom': 9}
dist: {'top': 4, 'left': 5, 'right': 0, 'bottom': 12}
dist: {'top': 2, 'left': 3, 'right': 3, 'bottom': 5}
dist: {'top': 2, 'left': 0, 'right': 6, 'bottom': 0}
dist: {'top': 2, 'left': 0, 'right':

## Save JSONs

In [ ]:
# import pandas as pd 
# from utils import *

# for mode in ['train', 'val', 'test']:
#     DATASET = get_coco(f'/Data_large/marine/Datasets/VENuS/annotations/perfect/{mode}.json')
#     annotations_x_band = pd.read_pickle(f'/Data_large/marine/Datasets/VENuS/annotations/perfect/{mode}_x_band_specialCase5.pkl')

#     for band in [5]:
#         DATASET['annotations'] = annotations_x_band[band]
#         save_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/{mode}_special_band_{band}.json'
#         save_to_json(DATASET, save_path)

#### Venus

In [ ]:
import pandas as pd 
from utils import *

for mode in ['train', 'val', 'test']:
    DATASET = get_coco(f'/Data_large/marine/Datasets/VENuS/annotations/perfect/{mode}.json')
    annotations_x_band = pd.read_pickle(f'/Data_large/marine/Datasets/VENuS/annotations/perfect/{mode}_x_band.pkl')

    for band in range(1,13,1):
        DATASET['annotations'] = annotations_x_band[band]
        save_path = f'/Data_large/marine/Datasets/VENuS/annotations/perfect/{mode}__band_{band}.json'
        save_to_json(DATASET, save_path)

#### Sentinel

In [38]:
import pandas as pd 
from utils import get_coco, save_to_json

senSave = '/Data_large/marine/Datasets/VDS2Raw/annotations'

band_index_to_tiff_index = {2:1, 3:2, 4:3, 8:4} # sentinel bands
band_indices = [2,3,4,8]

for mode in ['train', 'val', 'test']:
    DATASET = get_coco(f'{senSave}/{mode}.json')
    annotations_x_band = pd.read_pickle(f'{senSave}/{mode}_x_band.pkl')

    for band in band_indices:
        DATASET['annotations'] = annotations_x_band[band]
        save_path = f'{senSave}/{mode}__band_{band}.json'
        save_to_json(DATASET, save_path)

Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/train__band_2.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/train__band_3.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/train__band_4.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/train__band_8.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/val__band_2.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/val__band_3.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/val__band_4.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/val__band_8.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/test__band_2.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotations/test__band_3.json
Data successfully saved to /Data_large/marine/Datasets/VDS2Raw/annotat

# Thresholding SRC

In [ ]:
files = list_files('/Data_large/marine/PythonProjects/MMDET/studies/crops/B1')


for idx, f in enumerate(files[:3]):
    img = np.load(f)
    
    # visualize_img(img, size=1)
    img_x_method = {'original':img}
    for method in ('otsu', 'yen', 'isodata', 'li', 'mean', 'minimum', 'triangle', 'local'):
        print('Method:', method)
        img_x_method[method] = threshold(img, method=method)
        # visualize_img(threshold(img, method=method), size=1)
    
    if idx == 0:
        break

In [ ]:
import numpy as np
from skimage.morphology import reconstruction
from skimage.exposure import rescale_intensity
# Rescale image intensity so that we can see dim features.
img = rescale_intensity(img, in_range=(50, 200))
seed = np.copy(img)
seed[1:-1, 1:-1] = img.max()
mask = img

filled = reconstruction(seed, mask, method='erosion')

visualize_img(filled)

In [ ]:
start_idx = 3

for idx, f in enumerate(files[start_idx:]):
    img = np.load(f)
    
    # visualize_img(img, size=1)
    img_x_method = {'original':img}
    for method in ('otsu', 'yen', 'isodata', 'li', 'mean', 'minimum', 'triangle', 'local'):
        print('Method:', method)
        img_x_method[method] = threshold(img, method=method)
        # visualize_img(threshold(img, method=method), size=1)
    
    if idx == 0:
        break

fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(15,15))

c = 0
for i in range(3):
    for j in range(3):
        try:
            method = list(img_x_method.keys())[c]
            ax[i,j].imshow(img_x_method[method], cmap='jet')
            ax[i,j].set_title(method)
        except IndexError:
            print('End')
        
        c+=1

In [ ]:
BAND = 6
files = list_files(f'/Data_large/marine/PythonProjects/MMDET/studies/crops/B{BAND}')
files[10]

In [ ]:
BAND = 4
files = list_files(f'/Data_large/marine/PythonProjects/MMDET/studies/crops/B{BAND}')
files[10]

In [ ]:

def load_images(folder_path):
    png_files = [f for f in os.listdir(folder_path) if f.endswith('.png')]
    images = []
    for file in png_files:
        file_path = os.path.join(folder_path, file)
        image = Image.open(file_path)
        images.append(image)
    return images

def merge_images(images):
    width = max(image.width for image in images)
    total_height = sum(image.height for image in images)
    merged_image = Image.new('RGB', (width, total_height))
    y_offset = 0
    for image in images:
        merged_image.paste(image, (0, y_offset))
        y_offset += image.height
    return merged_image

def plot_thresholding_methods(BAND):
    methods_to_display = ['otsu', 'li', 'isodata', 'mean']
    files = list_files(f'/Data_large/marine/PythonProjects/MMDET/studies/crops/B{BAND}')
    fig, ax = plt.subplots(nrows=1, ncols=5, figsize=(12, 3), sharex=True, sharey=True, dpi=200)
    for idx, f in enumerate(files):
        img = np.load(f)
        img_x_method = {'original': img}
        for method in methods_to_display:
            img_x_method[method] = threshold(img, method=method)
        if idx == 0:
            break
    c = 0
    for j in range(len(methods_to_display)+1):
        try:
            method = list(img_x_method.keys())[c]
            ax[j].imshow(img_x_method[method], cmap='jet')
            ax[j].set_title(method.capitalize())
            ax[j].set_xlabel('x', fontsize=20, fontfamily='sans-serif', rotation=0, labelpad=10, va='center', ha='center', fontweight='normal', )
            ax[j].yaxis.set_label_position('right')
            ax[j].yaxis.tick_right()
            ax[j].set_ylabel('y', fontsize=20, fontfamily='sans-serif', rotation=0, labelpad=10, va='center', ha='center', fontweight='normal', )
            ax[j].yaxis.set_major_locator(MaxNLocator(nbins=5))
            ax[j].xaxis.set_major_locator(MaxNLocator(nbins=5))
            if j == 0:
                im = ax[j].imshow(img_x_method[method], cmap='jet')
                im.set_clim(50, 200)
                cbar = fig.colorbar(im, ax=ax[j], shrink=1, location='left')
                cbar.ax.tick_params(labelsize=15)
                cbar.ax.yaxis.set_label_coords(-1.5, 0.5)
        except IndexError:
            continue
        c += 1
    fig.savefig(f'/Data_large/marine/PythonProjects/MMDET/plots/thresholding_methods_B{BAND}.png', dpi=500, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

# Load images
folder_path = '/Data_large/marine/PythonProjects/MMDET/plots'
images = load_images(folder_path)

# Plot thresholding methods for each band
for BAND in range(1, 13):
    plot_thresholding_methods(BAND)

# Merge images
# merged_image = merge_images(images)
# merged_image.save('/Data_large/marine/PythonProjects/MMDET/plots/merged_image.png')